In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
from halcyon.Simulators.GNWSimRunner import GNWSimRunner
from halcyon.Models.PhoenixModels import PhoenixOnlyAddLayers, PhoenixAddAndProdLayers
from halcyon.Models.CellBox import CellBox
from halcyon.Models.MechanisticModel import PyTorchODE
from halcyon.Models.HybridModel import HybridModel
from halcyon.Trainer import Trainer
from halcyon.FittedTrainerList import FittedTrainerList
from halcyon.Losses import MSELossPlusMachineLearningL2
from copy import deepcopy
from halcyon.Simulators.GNWSimRunner import SimulationResult

# import torch.multiprocessing as mp
import multiprocessing as mp

from halcyon.Trainer import run_training_job

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import logging

import torch
import numpy as np
import random

seed = 174
# for CPU
torch.manual_seed(seed)              
# for all GPUs
torch.cuda.manual_seed_all(seed)     
np.random.seed(seed) 
random.seed(seed) 


logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True  # Important in Jupyter to reset config
)

# Simulating data

In [ ]:
# nr_genes_list = [32, 64, 128]
nr_genes_list = [15]
# nr_modules_list = [16, 5, 10]
nr_modules_list = [3]
sparsity_list = [0.05]
out_path = Path("../data/hybrid_model_first_tests") 

sim_outputs: list[SimulationResult] = []
for nr_genes in nr_genes_list: 
    for nr_modules in nr_modules_list: 
        for sparsity in sparsity_list: 
            gnw_sim = GNWSimRunner(
                settings_path=Path("../setting_files/gnw_testing_runs_settings.txt"),
            )
            generated_data_path = out_path / f"genes_{nr_genes}_modules_{nr_modules}_sparsity_{sparsity}"
            generated_data_path.mkdir(parents=True, exist_ok=True)
            network_name = 'my_network'
            sim_output = gnw_sim.run_simulation(
                n_genes=nr_genes,
                sparsity=sparsity,
                num_modules=nr_modules,
                network_name=network_name,
                output_path=generated_data_path,
                keep_self_interactions=False
            )
            sim_outputs.append(sim_output) 
sim_outputs

In [ ]:
sim_output = sim_outputs[0]
sim_output.network.plot()
plt.show()

# Trainers (with different models) creation

In [ ]:
train_jobs : list[Trainer] = []

## Mech model

In [ ]:
# Generate MM from this
mm = PyTorchODE.from_sbml(sim_output.data_path / 'my_network.xml', condition_names=(), from_gnw=True, device='cpu')
# Select only genes from specific module: here Nodes: 0, 11, 5, 8
mm.select_genes([f'Node_{i}' for i in [0, 11, 5, 8]])
# Fit MM on this data (skip for now?)
print(mm.modelled_gene_labels)
# Creating MM trainer even though it cannot be trained
mm_trainer = Trainer(
    mm,
    sim_output,
    batch_size=32,
    nr_training_steps=0,
    # optimizer_fn=lambda x: None
)

train_jobs.append(mm_trainer)
# mm_trainer.save(mm_trainer.save_path)

## ML and Hybrid model

In [ ]:
expr_data = sim_output.get_no_noise_ko_timeseries_expressiondata()
# Create ML model with and without forced_positive
models_to_use = []

ml_model = PhoenixAddAndProdLayers(
            modelled_gene_labels=expr_data.genes,
            hidden_size=5,
            condition_names=expr_data.all_environments,
            synthesis_only_positive=True,
            device='cuda:1'
    )

# Create hybrid model with MM and cellbox with add and prod
hybrid_model = HybridModel(deepcopy(mm), deepcopy(ml_model), device='cpu')

for model in [ml_model, hybrid_model]:
    # my_loss = MSELossPlusMachineLearningL2(lambda_param=0.5)
    trainer = Trainer(
        model,
        sim_output,
        batch_size=32,
        nr_training_steps=2,
    )
    train_jobs.append(trainer)

In [ ]:
train_jobs[0]

In [ ]:
for job in train_jobs:
    print(job)

# Do training

In [ ]:
train_jobs[2].run_for_parallelisation_and_save()

In [ ]:
mp.set_start_method("spawn", force=True)  
with mp.Pool(processes=3) as pool:
    pool.map(run_training_job, train_jobs)

# Plot results

In [ ]:
train_jobs[0:3]

In [ ]:
result_list = FittedTrainerList(train_jobs[0:3])

In [ ]:
# Currently loading hybrid / mechanistic model broken so will have to check this later. 
# The above implementation work (but doesnt load from saved files)
result_list = FittedTrainerList.from_train_jobs(train_jobs)

## Single KO

In [ ]:
result_list.plot_all_model_fits_into_one_fig(ymin=-.1);plt.tight_layout();plt.show()

## Double KO (unseen)

In [ ]:
result_list.plot_all_model_fits_into_one_fig(only_double_ko=True, ymin=-.1);plt.show()